In [19]:
import os
from bs4 import BeautifulSoup
import requests
import json
import re

In [22]:


def get_html_content(url, output_filename="downloaded_page.html"):
    """Fetches HTML from a URL and saves it to a local file."""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        print(f"Fetching content from: {url}...")
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        return soup
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching the URL: {e}")
        return None

def write_html_to_file(html_content, output_filename="downloaded_page.html"):
    """Saves provided HTML content to a local file."""
    try:
        with open(output_filename, "w", encoding="utf-8") as file:
            file.write(html_content.prettify())
        absolute_path = os.path.abspath(output_filename)
    except IOError as e:
        print(f"An error occurred while writing to the file: {e}")

def write_dictionary_to_json_file(data_dict, output_filename="output_data.json"):
    """Saves a Python dictionary to a JSON file."""
    try:
        with open(output_filename, "w", encoding="utf-8") as json_file:
            json.dump(data_dict, json_file, indent=4)
        absolute_path = os.path.abspath(output_filename)
    except IOError as e:
        print(f"An error occurred while writing to the JSON file: {e}")




# ABB

This converts the html conent into a json/pyton dictionary containing all the page data. 
It is this data that can then be used to parse out specific product specific information we require.

In [43]:
html_content = get_html_content("https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f")
write_html_to_file(html_content, "downloaded_page.html")

Fetching content from: https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f...


In [ ]:
def extract_abb_page_viewmodel_from_soup(soup, export_to_json=False):
    """Parses a BeautifulSoup object to extract the embedded JavaScript 'model'

    variable as a live Python dictionary.
    """
    if not soup:
        print("Invalid Soup object provided.")
        return None

    target_script = None
    for script in soup.find_all("script"):
        if script.string and "var model =" in script.string:
            target_script = script.string
            break
    if not target_script:
        print("Could not find the 'model' variable block in the HTML.")
        return None
    match = re.search(
        r"var model\s*=\s*(\{.*?\}*?\});\s*jsLibs\.push",
        target_script,
        re.DOTALL,
    )

    if not match:
        print("Regex failed to cleanly isolate the model JSON object boundary.")
        return None
    try:
        json_string = match.group(1)
        model_dict = json.loads(json_string)
        if export_to_json:
            write_dictionary_to_json_file(model_dict, "abb_page_viewmodel_data.json")
        return model_dict
    except json.JSONDecodeError as e:
        print(f"Failed to parse the extracted string into valid JSON: {e}")
        return None

html_content = get_html_content("https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f")
extracted_model = extract_abb_page_viewmodel_from_soup(html_content, export_to_json=True)
extracted_model
write_dictionary_to_json_file(extracted_model, "abb_page_viewmodel_data.json")

Fetching content from: https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f...


In [42]:
def extract_abb_product_info_from_viewmodel(model_dict):
   """Extracts specific product information from the ABB page viewmodel dictionary."""
   attribute_groups = model_dict.get("ProductViewModel", {}).get("Product", {}).get("attributeGroups", {}).get("items", [])
   for i, attribute_group in enumerate(attribute_groups):
       attribute_to_attribute_data = attribute_group.get("attributes", {})
       for attribute, attribute_data in attribute_to_attribute_data.items():
           print(f"Group {i} - Attribute: {attribute}, Value: {attribute_data.get('value')}")
       
   return attribute_groups

extracted_product_info = extract_abb_product_info_from_viewmodel(extracted_model)
extracted_product_info

Group 0 - Attribute: AbbEcoSolutions, Value: None
Group 0 - Attribute: AbbSitMeeGroWasLanTar, Value: None
Group 0 - Attribute: EndOfLifDisIns, Value: None
Group 0 - Attribute: EnvProDecEpd, Value: None
Group 0 - Attribute: ExtProLif, Value: None
Group 0 - Attribute: RecRatProAccToEn45555, Value: None
Group 1 - Attribute: ReachDeclaration, Value: None
Group 1 - Attribute: RoHSInformation, Value: None
Group 1 - Attribute: RoHSStatus, Value: None
Group 1 - Attribute: ConMinRepTem, Value: None
Group 1 - Attribute: ToxSubConAct, Value: None
Group 1 - Attribute: Scip, Value: None
Group 1 - Attribute: SimplifiedScip, Value: None
Group 2 - Attribute: EnvInf, Value: None
Group 3 - Attribute: LocOrdCodUSCON, Value: None
Group 3 - Attribute: ENumberFI, Value: None
Group 3 - Attribute: ENumberNO, Value: None
Group 3 - Attribute: Ean, Value: None
Group 3 - Attribute: MinimumOrderQuantity, Value: None
Group 3 - Attribute: CustomsTariffNumber, Value: None
Group 4 - Attribute: ProductNetWidth, Value: 

[{'code': 'AbbEcoSolutions',
  'description': 'ABB EcoSolutions',
  'visible': True,
  'attributes': {'AbbEcoSolutions': {'type': 'Enum',
    'attributeCode': 'AbbEcoSolutions',
    'attributeName': 'ABB EcoSolutions',
    'values': [{'text': 'Yes', 'enumerationValueCode': 'Y'}],
    'isInternal': False,
    'internal': False,
    'highlight': False},
   'AbbSitMeeGroWasLanTar': {'type': 'Enum',
    'attributeCode': 'AbbSitMeeGroWasLanTar',
    'attributeName': 'ABB Site Meeting Group Waste To Landfill Target',
    'values': [{'text': 'UL 2799 Zero Waste To Landfill Validation available',
      'enumerationValueCode': 'Ul2788ZeWaToLaVaAv'}],
    'isInternal': False,
    'internal': False,
    'highlight': False},
   'EndOfLifDisIns': {'type': 'Text',
    'attributeCode': 'EndOfLifDisIns',
    'attributeName': 'End Of Life Disassembling Instructions',
    'values': [{'text': '9AKK108468A2348',
      'link': {'text': '9AKK108468A2348',
       'documentId': '9AKK108468A2348',
       'type